# Build Matching Dataset for Label Spreading — COCO Train (hard GT only)

Loads Grounding DINO predictions (parquet) for the COCO **train2017** split
and matches them against the standard COCO GT (`instances_train2017.json`)
only. There is no soft-labeled COCO GT for the train split, so — unlike
`build_matching_dataset.ipynb` (val), which matches against both a soft GT
and the standard COCO GT and keeps them as two independent columns — this
notebook has a single ground truth and bakes the hard-GT match directly into
`soft_label` / `label` / `score` / `gdino_match` (one-hot). The output, `data/matched_dataset/coco_train_matched.parquet`, is
the input of the COCO pseudo-label generation in `pseudo_labels/`.

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pycocotools.coco import COCO

# ── Paths ─────────────────────────────────────────────────────────────────────
PARQUET_PATH = "../inference/predictions/gdino/coco_train/predictions.parquet"
COCO_GT_PATH = "/home/klees/SoftTeacher/data/coco/annotations/instances_train2017.json"
OUTPUT_PATH  = "../data/matched_dataset/coco_train_matched.parquet"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ── Matching parameters ───────────────────────────────────────────────────────
IOU_THR   = 0.5    # IoU threshold for a detection to be considered a match

# Train predictions are 300 unfiltered boxes/image across 118k images (46GB,
# 35.5M rows) — unlike the val notebook (which keeps SCORE_THR=0 and filters
# downstream), pre-filter here via parquet predicate
# pushdown so the matching loop and the embedding column never materialize
# the ~33.5M rows that would be discarded downstream anyway.
SCORE_THR = 0.2

N_CLASSES = 80     # COCO classes
BG_LABEL  = 80      # index used for unmatched / background predictions

## 1  Load data

In [ ]:
df_pred = pd.read_parquet(PARQUET_PATH, filters=[("score", ">=", SCORE_THR)])
print(f"Predictions (score >= {SCORE_THR}): {len(df_pred):,} detections across "
      f"{df_pred['image_id'].nunique():,} images")

# ── Standard COCO GT (train2017) ──────────────────────────────────────────────
coco_gt = COCO(COCO_GT_PATH)

coco_cats       = coco_gt.dataset["categories"]                          # ordered list
coco_id_to_name = {c["id"]: c["name"] for c in coco_cats}
coco_id_to_idx  = {c["id"]: i         for i, c in enumerate(coco_cats)}  # cat ID → 0-79

print(f"COCO train GT       : {len(coco_gt.dataset['annotations']):,} annotations "
      f"across {len(coco_gt.dataset['images']):,} images")

## 2  IoU matching helpers

In [ ]:
def iou_matrix(boxes1, boxes2):
    """(N,4) × (M,4) [x1,y1,x2,y2] → (N,M) IoU matrix."""
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)))
    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])
    inter  = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    areas1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    areas2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union  = areas1[:, None] + areas2[None, :] - inter
    return inter / np.maximum(union, 1e-6)


def greedy_match(pred_scores, iou_mat, iou_thr):
    """Greedy 1-1 matching sorted by descending score.

    Returns
    -------
    match_iou : (N,) float  – IoU of the matched GT (0 if unmatched)
    match_idx : (N,) int    – index into GT list (-1 if unmatched)
    """
    N, M = iou_mat.shape
    matched_gt = np.zeros(M, dtype=bool)
    match_iou  = np.zeros(N, dtype=np.float32)
    match_idx  = np.full(N, -1, dtype=int)
    for i in np.argsort(pred_scores)[::-1]:
        row = iou_mat[i].copy()
        row[matched_gt] = 0.0
        best = int(row.argmax())
        if row[best] >= iou_thr:
            match_iou[i] = row[best]
            match_idx[i] = best
            matched_gt[best] = True
    return match_iou, match_idx

## 3  Run per-image matching

For each image, match GDino predictions → standard COCO GT annotations
(IoU ≥ threshold, greedy by score) → `coco_id`, `category_id`,
`category_label`, then fold that single match directly into the one-hot
`soft_label` / `label` / `score` / `gdino_match` columns used
downstream (index 80 = unmatched/background,
all-zero — same convention as the val notebook's unmatched rows).

In [ ]:
ZERO_81 = np.zeros(N_CLASSES + 1, dtype=np.float32)   # 81-dim zero vector for unmatched rows

rows = []

for img_id, grp in tqdm(df_pred.groupby("image_id"), desc="Matching images"):

    # ── Prediction arrays ────────────────────────────────────────────────────
    pred_boxes  = np.array(grp["box"].tolist(), dtype=np.float32)   # (N,4) [x1,y1,x2,y2]
    pred_scores = grp["score"].values.astype(np.float32)
    pred_labels = grp["label"].values.astype(np.int32)              # COCO category IDs
    pred_embs   = grp["last_hidden_state"].tolist()                  # list of (256,) arrays
    N           = len(pred_boxes)

    # ── Match against standard COCO GT ───────────────────────────────────────
    coco_anns      = coco_gt.imgToAnns.get(img_id, [])
    coco_match_iou = np.zeros(N, dtype=np.float32)
    coco_match_idx = np.full(N, -1, dtype=int)

    if coco_anns:
        gt_xywh  = np.array([a["bbox"] for a in coco_anns], dtype=np.float32)
        gt_boxes = gt_xywh.copy()
        gt_boxes[:, 2] += gt_boxes[:, 0]
        gt_boxes[:, 3] += gt_boxes[:, 1]
        iou_mat  = iou_matrix(pred_boxes, gt_boxes)
        coco_match_iou, coco_match_idx = greedy_match(pred_scores, iou_mat, IOU_THR)

    # ── Build rows ───────────────────────────────────────────────────────────
    for i in range(N):
        gdino_cat_id   = int(pred_labels[i])
        gdino_cat_name = coco_id_to_name.get(gdino_cat_id, "unknown")

        c_idx   = int(coco_match_idx[i])
        matched = c_idx >= 0
        if matched:
            cann           = coco_anns[c_idx]
            coco_id        = float(cann["id"])
            category_id    = float(cann["category_id"])
            category_label = coco_id_to_name.get(cann["category_id"])
            cls_idx        = coco_id_to_idx[cann["category_id"]]
            bbox           = cann["bbox"]                        # already COCO [x,y,w,h]
        else:
            coco_id        = None
            category_id    = None
            category_label = None
            cls_idx        = None
            bbox           = None

        # One-hot supervision straight from the hard COCO GT match (there's no
        # soft label to blend with here — see the markdown above).
        soft_label = ZERO_81.copy()
        if matched:
            soft_label[cls_idx] = 1.0
        label = cls_idx if matched else BG_LABEL

        rows.append({
            # ── GDino prediction ─────────────────────────
            "image_id"       : img_id,
            "gdino_pred_id"  : i + 1,
            "gdino_bbox"     : pred_boxes[i].tolist(),
            "gdino_score"    : float(pred_scores[i]),
            "gdino_label"    : gdino_cat_name,
            "gdino_embedding": pred_embs[i],               # (256,) float32
            # ── Hard COCO GT match ────────────────────────
            "gdino_match"    : int(matched),
            "gdino_bbox_iou" : float(coco_match_iou[i]),
            "bbox"           : bbox,                       # [x,y,w,h] COCO format or None
            "score"          : float(matched),             # one-hot ⇒ max is 1.0 or 0.0
            "soft_label"     : soft_label,                 # (81,) one-hot — incl. background at [80]
            "label"          : label,                      # 0-79 COCO class or 80 background
            "coco_id"        : coco_id,
            "category_id"    : category_id,
            "category_label" : category_label,
        })

df = pd.DataFrame(rows)
print(f"\nDataset shape : {df.shape}")
print(f"COCO GT match : {df['gdino_match'].sum():,} / {len(df):,}  ({df['gdino_match'].mean():.1%})")

## 4  Inspect

In [ ]:
df.info()
df.head(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].hist(df["gdino_score"], bins=50, color="steelblue")
axes[0].set_xlabel("GDino score"); axes[0].set_ylabel("Count")
axes[0].set_title("Prediction score distribution")
axes[0].set_yscale("log"); axes[0].grid(True, ls=":")

matched_iou = df.loc[df["gdino_match"] == 1, "gdino_bbox_iou"]
axes[1].hist(matched_iou, bins=50, color="darkorange")
axes[1].set_xlabel("IoU (COCO GT match)"); axes[1].set_ylabel("Count")
axes[1].set_title(f"IoU distribution – matched ({len(matched_iou):,})")
axes[1].grid(True, ls=":")

top_cats = df.loc[df["gdino_match"] == 1, "category_label"].value_counts().head(15)
axes[2].barh(top_cats.index[::-1], top_cats.values[::-1], color="forestgreen")
axes[2].set_xlabel("Count"); axes[2].set_title("Top matched COCO categories")
axes[2].grid(True, ls=":")

plt.tight_layout()
plt.show()

## 5  Save

In [ ]:
df.to_parquet(OUTPUT_PATH, engine="pyarrow", compression="snappy", index=False)
print(f"Saved → {OUTPUT_PATH}")
print(f"  Rows    : {len(df):,}")
print(f"  Columns : {df.columns.tolist()}")
print(f"  Size    : {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")